In [10]:
import os
REPO_URL = "https://github.com/minyi-k03/VideoProcessing-Project.git"
if os.path.isdir("/content/repo/.git"):
    !git -C /content/repo pull --ff-only          # 이미 받아 둔 경우 최신으로
else:
    !git clone --branch YongMin --depth 1 {REPO_URL} /content/repo
!git -C /content/repo log --oneline -3             # 5f26848 이 보여야 함
!wget -q -O /content/basicsr_mf.py https://raw.githubusercontent.com/XPixelGroup/BasicSR/master/basicsr/utils/matlab_functions.py
!grep -n "^import\|^from" /content/basicsr_mf.py   # math, numpy, torch 외 import가 있으면 알려 주세요
assert os.path.isdir("/content/repo/src/sr_pipeline"), "clone 실패: REPO_URL/푸시 확인"
print("셀 1 OK")


Already up to date.
5f26848 (grafted, HEAD -> YongMin, origin/YongMin) test: bicubic 참조 헬퍼의 pad_start 음수 코너케이스 수정, 극단 축소 케이스 추가 (T1.2 Gemini 교차 리뷰 반영)
1:import math
2:import numpy as np
3:import torch
셀 1 OK


In [11]:
import sys, importlib, importlib.util, torch
sys.path.insert(0, "/content/repo/src")
sys.path_importer_cache.pop("/content/repo/src", None)   # 클론 전에 import를 시도했던 커널의 "경로 없음" 캐시 제거
importlib.invalidate_caches()
from sr_pipeline.common.bicubic import bicubic_resize
spec = importlib.util.spec_from_file_location("mf", "/content/basicsr_mf.py")
mf = importlib.util.module_from_spec(spec); spec.loader.exec_module(mf)
assert torch.cuda.is_available(), "GPU 런타임이 아님"
dev = torch.device("cuda")

g = torch.Generator().manual_seed(0)
cases = [(64,64,0.25,True),(64,64,0.25,False),(37,53,0.25,True),
         (16,16,4.0,True),(23,31,4.0,True),(180,320,4.0,True),(720,1280,0.25,True)]
all_ok = True
basicsr_compared = 0
for h, w, s, aa in cases:
    x = torch.rand(1, 3, h, w, generator=g)
    cpu = bicubic_resize(x, s, antialiasing=aa)[0]
    gpu = bicubic_resize(x.to(dev), s, antialiasing=aa)[0].cpu()
    e_gpu = (gpu - cpu).abs().max().item()
    x64 = x.double()
    e64 = (bicubic_resize(x64.to(dev), s, antialiasing=aa)[0].cpu() - bicubic_resize(x64, s, antialiasing=aa)[0]).abs().max().item()
    gpu_ok = e_gpu < 1e-5 and e64 < 1e-10

    # BasicSR 참조(내부 float32). BasicSR 자체가 예외를 내는 조합은 비교만 건너뛴다(우리 쪽 실패로 세지 않음).
    try:
        ref = mf.imresize(x[0].clone(), s, antialiasing=aa)
        e_ref = (cpu - ref).abs().max().item()
        ref_ok = ref.shape == cpu.shape and e_ref < 2e-5
        ref_txt = f"vs BasicSR={e_ref:.2e}"
        basicsr_compared += 1
    except RuntimeError as err:
        ref_ok = True
        ref_txt = f"vs BasicSR=건너뜀(BasicSR 자체 오류: {str(err)[:50]})"

    good = gpu_ok and ref_ok
    all_ok &= good
    print(f"{h}x{w} x{s} aa={aa}: shape={tuple(cpu.shape)} {ref_txt} cuda-cpu={e_gpu:.2e} cuda-cpu(fp64)={e64:.2e} {'OK' if good else 'FAIL'}")
print(f"BasicSR와 비교한 케이스: {basicsr_compared}/{len(cases)}")
print("전체:", "PASS" if all_ok else "FAIL")


64x64 x0.25 aa=True: shape=(3, 16, 16) vs BasicSR=1.79e-07 cuda-cpu=1.19e-07 cuda-cpu(fp64)=0.00e+00 OK
64x64 x0.25 aa=False: shape=(3, 16, 16) vs BasicSR=건너뜀(BasicSR 자체 오류: The size of tensor a (0) must match the size of te) cuda-cpu=0.00e+00 cuda-cpu(fp64)=0.00e+00 OK
37x53 x0.25 aa=True: shape=(3, 10, 14) vs BasicSR=1.19e-07 cuda-cpu=1.79e-07 cuda-cpu(fp64)=0.00e+00 OK
16x16 x4.0 aa=True: shape=(3, 64, 64) vs BasicSR=2.38e-07 cuda-cpu=0.00e+00 cuda-cpu(fp64)=0.00e+00 OK
23x31 x4.0 aa=True: shape=(3, 92, 124) vs BasicSR=2.38e-07 cuda-cpu=2.38e-07 cuda-cpu(fp64)=0.00e+00 OK
180x320 x4.0 aa=True: shape=(3, 720, 1280) vs BasicSR=3.58e-07 cuda-cpu=1.19e-07 cuda-cpu(fp64)=0.00e+00 OK
720x1280 x0.25 aa=True: shape=(3, 180, 320) vs BasicSR=2.38e-07 cuda-cpu=1.19e-07 cuda-cpu(fp64)=0.00e+00 OK
BasicSR와 비교한 케이스: 6/7
전체: PASS


In [12]:
import time
# (a) inference_mode에서 먼저 호출한 뒤 CUDA 역전파
with torch.inference_mode():
    s = torch.rand(1, 3, 32, 32, device=dev)
    bicubic_resize(s, 0.25); bicubic_resize(s, 4.0)
x = torch.rand(1, 3, 32, 32, device=dev, requires_grad=True)
(bicubic_resize(x, 0.25).sum() + bicubic_resize(x, 4.0).sum()).backward()
print("backward after inference_mode on cuda:", torch.isfinite(x.grad).all().item())

# (b) AMP 안에서 dtype (docstring의 "FP16으로 내려간다" 주장 확인)
with torch.autocast("cuda", dtype=torch.float16):
    print("autocast 안 출력 dtype:", bicubic_resize(torch.rand(1, 3, 32, 32, device=dev), 4.0).dtype)

# (c) 추론 시간 프로토콜(320x180 -> 1280x720, 워밍업 10 + 100회), 캐시 적중 상태
x = torch.rand(1, 3, 180, 320, device=dev)
with torch.inference_mode():
    for _ in range(10): bicubic_resize(x, 4.0)
    torch.cuda.synchronize(); t = time.perf_counter()
    for _ in range(100): bicubic_resize(x, 4.0)
    torch.cuda.synchronize()
print(f"bicubic x4 평균: {(time.perf_counter() - t) / 100 * 1000:.3f} ms")


backward after inference_mode on cuda: True
autocast 안 출력 dtype: torch.float16
bicubic x4 평균: 0.641 ms
